# **Week 7 – Outlier Detection and Data Quality**

goal : 
- Flag and/or remove outliers: price, price per square footage, close to list ratio, or days on market 

### Set Up

In [1]:
from pathlib import Path
from datetime import datetime
import re
import pandas as pd
import os
import numpy as np

DATA_DIR = Path("/Users/kmaxx/Desktop/IDX-da/idx_data")

sold_df = pd.read_csv(DATA_DIR / "featured_sold.csv", low_memory= False)
listing_df = pd.read_csv(DATA_DIR / "featured_listing.csv", low_memory= False)

### Outlier Detection for Sold df

In [2]:
living_area = pd.to_numeric(
    sold_df["LivingArea"],
    errors="coerce"
)

valid_living_area = living_area.notna()
under_80 = valid_living_area & (living_area < 80)

under_80_count = under_80.sum()
valid_count = valid_living_area.sum()
under_80_percentage = under_80_count / valid_count * 100

print(f"Records with valid LivingArea: {valid_count:,}")
print(f"Records below 80 sq ft: {under_80_count:,}")
print(f"Percentage below 80 sq ft: {under_80_percentage:.4f}%")

Records with valid LivingArea: 447,491
Records below 80 sq ft: 5
Percentage below 80 sq ft: 0.0011%


In [3]:
review_columns = [
    "ListingKey",
    "CountyOrParish",
    "PropertySubType",
    "ClosePrice",
    "LivingArea",
    "PricePerSqFt"
]

display(
    sold_df.loc[
        under_80,
        review_columns
    ].sort_values("LivingArea")
)


,ListingKey,CountyOrParish,PropertySubType,ClosePrice,LivingArea,PricePerSqFt
66823,1046524604,Ventura,Condominium,362000.0,1.0,362000.0
130182,1052366102,San Francisco,NaN,590000.0,1.0,590000.0
329438,1052326217,San Francisco,NaN,735000.0,1.0,735000.0
43667,1076125889,Kern,Cabin,284000.0,2.0,142000.0
159589,1076248355,Ventura,SingleFamilyResidence,950000.0,2.0,475000.0


In [9]:
close_price = pd.to_numeric(
    sold_df["ClosePrice"],
    errors="coerce"
)

low_price_mask = close_price < 10_000

low_price_df = sold_df.loc[
    low_price_mask,
    [
        "ListingKey",
        "CountyOrParish",
        "City",
        "PropertyType",
        "PropertySubType",
        "ClosePrice",
        "OriginalListPrice",
        "LivingArea",
        "CloseDate"
    ]
].copy()

low_price_df["ClosePrice"] = close_price.loc[low_price_mask]

print(f"Records with ClosePrice below $10,000: {len(low_price_df):,}")
print(
    f"Percentage of valid ClosePrice records: "
    f"{len(low_price_df) / close_price.notna().sum() * 100:.4f}%"
)

display(
    low_price_df.sort_values("ClosePrice").head(100)
)

Records with ClosePrice below $10,000: 15
Percentage of valid ClosePrice records: 0.0034%


,ListingKey,CountyOrParish,City,PropertyType,PropertySubType,ClosePrice,OriginalListPrice,LivingArea,CloseDate
347567,1077504198,Los Angeles,West Hills,Residential,SingleFamilyResidence,1.15,1249000.0,1811.0,2024-12-09
317508,1144223437,Riverside,Palm Desert,Residential,SingleFamilyResidence,1.75,1795000.0,3513.0,2026-01-30
176084,1059929060,Contra Costa,Brentwood,Residential,SingleFamilyResidence,345.00,784345.0,2048.0,2024-05-15
28520,1076621601,Riverside,Desert Hot Springs,Residential,SingleFamilyResidence,380.00,380000.0,1200.0,2024-09-03
335637,1095537331,Alameda,Alameda,Residential,Condominium,464.00,866464.0,1102.0,2024-12-27
213213,1129501416,Riverside,Cathedral City,Residential,SingleFamilyResidence,485.00,498000.0,1305.0,2025-10-23
213096,1129896415,Ventura,Ventura,Residential,ManufacturedOnLand,500.00,510000.0,1400.0,2025-10-27
394147,1075040391,Los Angeles,Duarte,Residential,Townhouse,675.00,675000.0,1296.0,2024-07-02
117457,1093752049,San Bernardino,Big Bear,Residential,Timeshare,4500.00,9900.0,2045.0,2025-08-15
289715,1149541990,San Bernardino,Big Bear Lake,Residential,Timeshare,5000.00,5000.0,2880.0,2026-02-03


In [5]:
#Sold_df
import pandas as pd

outlier_columns = [
    "ClosePrice",
    "LivingArea",
    "DaysOnMarket"
]

lower_bounds = {
    "ClosePrice": 10_000,
    "LivingArea": 80,
    "DaysOnMarket": 0
}

sold_flagged_df = sold_df.copy()

sold_flagged_df[outlier_columns] = sold_flagged_df[
    outlier_columns
].apply(pd.to_numeric, errors="coerce")

outlier_flag_columns = []
outlier_summary = []

for column in outlier_columns:
    q1 = sold_flagged_df[column].quantile(0.25)
    q3 = sold_flagged_df[column].quantile(0.75)
    iqr = q3 - q1

    lower_bound = lower_bounds[column]
    upper_bound = q3 + 3 * iqr

    flag_column = f"{column}_Outlier"
    outlier_flag_columns.append(flag_column)

    sold_flagged_df[flag_column] = (
        sold_flagged_df[column].notna()
        & ~sold_flagged_df[column].between(
            lower_bound,
            upper_bound
        )
    )

    outlier_summary.append({
        "Variable": column,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "LowerBound": lower_bound,
        "UpperBound": upper_bound,
        "OutlierCount": sold_flagged_df[flag_column].sum(),
        "OutlierPercentage": (
            sold_flagged_df[flag_column].mean() * 100
        )
    })

outlier_summary = pd.DataFrame(outlier_summary).round(2)

display(outlier_summary)

,Variable,Q1,Q3,IQR,LowerBound,UpperBound,OutlierCount,OutlierPercentage
0,ClosePrice,575000.0,1300000.0,725000.0,10000,3475000.0,14322,3.20
1,LivingArea,1248.0,2224.0,976.0,80,5152.0,4931,1.10
2,DaysOnMarket,8.0,48.0,40.0,0,168.0,11887,2.66


In [6]:
# Keep all rows and add one combined outlier flag
sold_flagged_df["AnyOutlier"] = sold_flagged_df[
    outlier_flag_columns
].any(axis=1)

# Remove rows flagged in at least one variable
sold_no_outliers_df = sold_flagged_df.loc[
    ~sold_flagged_df["AnyOutlier"]
].copy()

print(f"Original rows: {len(sold_df):,}")
print(f"Flagged rows: {sold_flagged_df['AnyOutlier'].sum():,}")
print(f"Rows after removal: {len(sold_no_outliers_df):,}")

Original rows: 447,491
Flagged rows: 26,526
Rows after removal: 420,965


#### Outliers for the features

In [15]:
import numpy as np
import pandas as pd

derived_columns = [
    "PriceRatio",
    "PricePerSqFt"
]

sold_derived_flagged_df = sold_no_outliers_df.copy()

sold_derived_flagged_df[derived_columns] = sold_derived_flagged_df[
    derived_columns
].apply(pd.to_numeric, errors="coerce").replace(
    [np.inf, -np.inf],
    np.nan
)

derived_flag_columns = []
derived_outlier_summary = []

for column in derived_columns:
    values = sold_derived_flagged_df[column].dropna()

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1

    if column == "PriceRatio":
        lower_bound = q1 - 3 * iqr
    else:
        lower_bound = 50

    upper_bound = q3 + 3 * iqr

    flag_column = f"{column}_Outlier"
    derived_flag_columns.append(flag_column)

    sold_derived_flagged_df[flag_column] = (
        sold_derived_flagged_df[column].notna()
        & ~sold_derived_flagged_df[column].between(
            lower_bound,
            upper_bound
        )
    )

    derived_outlier_summary.append({
        "Variable": column,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "LowerBound": lower_bound,
        "UpperBound": upper_bound,
        "OutlierCount": sold_derived_flagged_df[flag_column].sum(),
        "OutlierPercentage": (
            sold_derived_flagged_df[flag_column].mean() * 100
        )
    })

derived_outlier_summary = pd.DataFrame(
    derived_outlier_summary
).round(2)

display(derived_outlier_summary)

,Variable,Q1,Q3,IQR,LowerBound,UpperBound,OutlierCount,OutlierPercentage
0,PriceRatio,0.96,1.02,0.06,0.77,1.21,11758,2.79
1,PricePerSqFt,366.67,713.74,347.07,50.00,1754.95,2552,0.61


In [16]:
sold_derived_flagged_df["AnyDerivedOutlier"] = (
    sold_derived_flagged_df[derived_flag_columns].any(axis=1)
)

sold_final_no_outliers_df = sold_derived_flagged_df.loc[
    ~sold_derived_flagged_df["AnyDerivedOutlier"]
].copy()

print(f"Starting rows: {len(sold_no_outliers_df):,}")
print(
    f"PriceRatio outliers: "
    f"{sold_derived_flagged_df['PriceRatio_Outlier'].sum():,}"
)
print(
    f"PricePerSqFt outliers: "
    f"{sold_derived_flagged_df['PricePerSqFt_Outlier'].sum():,}"
)
print(
    f"Total rows flagged: "
    f"{sold_derived_flagged_df['AnyDerivedOutlier'].sum():,}"
)
print(f"Rows retained: {len(sold_final_no_outliers_df):,}")

Starting rows: 420,965
PriceRatio outliers: 11,758
PricePerSqFt outliers: 2,552
Total rows flagged: 13,859
Rows retained: 407,106


In [17]:
review_columns = [
    "ListingKey",
    "CountyOrParish",
    "PropertySubType",
    "ClosePrice",
    "LivingArea",
    "PriceRatio",
    "PricePerSqFt",
    "PriceRatio_Outlier",
    "PricePerSqFt_Outlier",
    "AnyDerivedOutlier"
]

display(
    sold_derived_flagged_df.loc[
        sold_derived_flagged_df["AnyDerivedOutlier"],
        review_columns
    ].sort_values(
        "PricePerSqFt",
        ascending=False
    ).head(100)
)

,ListingKey,CountyOrParish,PropertySubType,ClosePrice,LivingArea,PriceRatio,PricePerSqFt,PriceRatio_Outlier,PricePerSqFt_Outlier,AnyDerivedOutlier
73077,1075621254,Los Angeles,SingleFamilyResidence,1862935.0,100.0,0.931468,18629.350000,False,True,True
191487,1104106961,Los Angeles,SingleFamilyResidence,1640000.0,100.0,0.964706,16400.000000,False,True,True
346710,1079443527,Los Angeles,Triplex,1600000.0,100.0,0.810127,16000.000000,False,True,True
216312,1120354181,Los Angeles,SingleFamilyResidence,643500.0,105.0,0.919286,6128.571429,False,True,True
102480,1130424713,Santa Cruz,SingleFamilyResidence,3333333.0,546.0,NaN,6105.005495,False,True,True
...,...,...,...,...,...,...,...,...,...,...
7934,1061954228,San Diego,SingleFamilyResidence,2725000.0,913.0,0.990909,2984.665936,False,True,True
48735,1103007379,Santa Clara,SingleFamilyResidence,3412500.0,1146.0,1.000000,2977.748691,False,True,True
20428,1079431022,Santa Clara,SingleFamilyResidence,2380000.0,800.0,1.043860,2975.000000,False,True,True
178985,1112457147,Santa Clara,SingleFamilyResidence,2820000.0,950.0,1.132530,2968.421053,False,True,True


### Outlier Detection for Listing df

In [13]:
listing_price = pd.to_numeric(
    listing_df["ListPrice"],
    errors="coerce"
)

low_price_mask = listing_price < 10_000

low_price_listings = listing_df.loc[
    low_price_mask,
    [
        "ListingKey",
        "CountyOrParish",
        "City",
        "PropertyType",
        "PropertySubType",
        "ListPrice",
        "OriginalListPrice",
        "LivingArea",
        "ListingContractDate"
    ]
].copy()

low_price_listings["ListPrice"] = listing_price.loc[low_price_mask]

valid_count = listing_price.notna().sum()
low_price_count = low_price_mask.sum()
low_price_percentage = low_price_count / valid_count * 100

print(f"Valid ListPrice records: {valid_count:,}")
print(f"ListPrice below $10,000: {low_price_count:,}")
print(f"Percentage below $10,000: {low_price_percentage:.4f}%")

display(
    low_price_listings
    .sort_values("ListPrice")
    .head(100)
)

Valid ListPrice records: 613,615
ListPrice below $10,000: 29
Percentage below $10,000: 0.0047%


,ListingKey,CountyOrParish,City,PropertyType,PropertySubType,ListPrice,OriginalListPrice,LivingArea,ListingContractDate
204511,1168684489,Tuolumne,Sonora,Residential,SingleFamilyResidence,1.0,1.0,1494.0,2026-05-19
530644,1106999145,San Bernardino,Apple Valley,Residential,SingleFamilyResidence,100.0,100.0,2866.0,2025-02-11
530574,1107004196,San Bernardino,Oak Hills,Residential,SingleFamilyResidence,100.0,1.0,3479.0,2025-02-11
408690,1112544483,San Bernardino,Crestline,Residential,SingleFamilyResidence,100.0,100.0,1440.0,2025-05-02
297921,1065201431,Orange,Anaheim,Residential,SingleFamilyResidence,695.0,695.0,1170.0,2024-01-01
495448,1109236257,San Bernardino,Hesperia,Residential,SingleFamilyResidence,1000.0,1000.0,2793.0,2025-03-25
427469,1144769805,San Diego,Escondido,Residential,Condominium,1500.0,2800.0,100.0,2025-10-27
38450,1093665827,Riverside,Palm Desert,Residential,Timeshare,2000.0,2000.0,800.0,2024-11-05
204360,1168716904,San Diego,Escondido,Residential,Condominium,2000.0,2000.0,100.0,2026-05-19
98672,1174704301,San Mateo,Daly City,Residential,SingleFamilyResidence,2500.0,2500.0,500.0,2026-06-16


In [10]:
# Listing_df
import pandas as pd

outlier_columns = [
    "ListPrice",
    "LivingArea",
    "DaysOnMarket"
]

lower_bounds = {
    "ListPrice": 10_000,
    "LivingArea": 80,
    "DaysOnMarket": 0
}

listing_flagged_df = listing_df.copy()

listing_flagged_df[outlier_columns] = listing_flagged_df[
    outlier_columns
].apply(pd.to_numeric, errors="coerce")

outlier_flag_columns = []
outlier_summary = []

for column in outlier_columns:
    q1 = listing_flagged_df[column].quantile(0.25)
    q3 = listing_flagged_df[column].quantile(0.75)
    iqr = q3 - q1

    lower_bound = lower_bounds[column]
    upper_bound = q3 + 3 * iqr

    flag_column = f"{column}_Outlier"
    outlier_flag_columns.append(flag_column)

    listing_flagged_df[flag_column] = (
        listing_flagged_df[column].notna()
        & ~listing_flagged_df[column].between(
            lower_bound,
            upper_bound
        )
    )

    outlier_summary.append({
        "Variable": column,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "LowerBound": lower_bound,
        "UpperBound": upper_bound,
        "OutlierCount": listing_flagged_df[flag_column].sum(),
        "OutlierPercentage": listing_flagged_df[flag_column].mean() * 100
    })

listing_outlier_summary = pd.DataFrame(outlier_summary).round(2)

display(listing_outlier_summary)

,Variable,Q1,Q3,IQR,LowerBound,UpperBound,OutlierCount,OutlierPercentage
0,ListPrice,580900.0,1380000.0,799100.0,10000,3777300.0,25615,4.17
1,LivingArea,1248.0,2303.0,1055.0,80,5468.0,9624,1.57
2,DaysOnMarket,5.0,22.0,17.0,0,73.0,25831,4.21


In [11]:
listing_flagged_df["AnyOutlier"] = listing_flagged_df[
    outlier_flag_columns
].any(axis=1)

listing_no_outliers_df = listing_flagged_df.loc[
    ~listing_flagged_df["AnyOutlier"]
].copy()

print(f"Original rows: {len(listing_df):,}")
print(f"Flagged rows: {listing_flagged_df['AnyOutlier'].sum():,}")
print(f"Rows after removal: {len(listing_no_outliers_df):,}")

Original rows: 613,615
Flagged rows: 51,637
Rows after removal: 561,978


### Export

In [18]:
sold_derived_flagged_df.to_csv(
    DATA_DIR / "sold_derived_outliers_flagged.csv",
    index=False
)

sold_final_no_outliers_df.to_csv(
    DATA_DIR / "sold_final_outliers_removed.csv",
    index=False
)

listing_flagged_df.to_csv(
    DATA_DIR / "listing_outliers_flagged.csv",
    index=False
)

listing_no_outliers_df.to_csv(
    DATA_DIR / "listing_outliers_removed.csv",
    index=False
)
